In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "key"

In [23]:
import random
import json

random.seed(42)

MAX_TIME = 1200

with open("interview_questions.json", "r") as f:
    questions = json.load(f)

# shuffle once
random.shuffle(questions)

selected_questions = []
total_time = 0

for q in questions:
    if total_time + q["time_seconds"] > MAX_TIME:
        continue

    selected_questions.append(q["question"])
    total_time += q["time_seconds"]

    if total_time >= MAX_TIME:
        break

print("Questions selected")

Questions selected


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", temperature=0, seed=42)


In [26]:
def run_interview(persona: dict, questions: list[str]):
    transcript = []

    for q in questions:
        transcript.append(("Interviewer", q))

        prompt = f"""
You are roleplaying as this person:

{json.dumps(persona, indent=2)}

Conversation so far:
{transcript}

Answer the latest question naturally as this person.
Do not explain. Just answer.
"""

        answer = llm.invoke(prompt).content.strip()

        transcript.append(("Interviewee", answer))

    return transcript
    

In [7]:
import json

with open("../data/alberti.json", "r") as f:
    persona_seed = json.load(f)

print(persona_seed)

{'uuid': 'e7c0574639a244c8972c92aab9501035', 'professional_persona': 'Mary Alberti is a front-line food service specialist whose razor-sharp cash handling, inventory tracking, and POS mastery combine with a disciplined, routine-driven work ethic, enabling them to calmly resolve high-pressure customer issues and hit performance targets while eyeing a promotion to shift supervisor.', 'sports_persona': 'Mary Alberti fuels their fitness routine by clocking 3-5 mile runs around Lake Mendota with the Madison Runners Club, roots for the Wisconsin Badgers basketball team in the winter, cheers the Milwaukee Brewers in summer, and never misses a Green Bay Packers game on Sundays, balancing competitive spirit with disciplined consistency.', 'arts_persona': 'Mary Alberti finds creative inspiration in the lyrical storytelling of John Prine, the atmospheric indie folk of Bon Iver, and the classic cinematography of Wes Anderson, often attending local art walks and museum exhibits to unwind after a sh

In [28]:
t = run_interview(persona_seed, selected_questions)


In [29]:
from pathlib import Path

def save_transcript(transcript, filepath):
    data = [
        {"speaker": speaker, "text": text}
        for speaker, text in transcript
    ]

    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w") as f:
        json.dump(data, f, indent=2)

save_transcript(t, "interview_transcripts/alberti_interview_transcript.json")

In [17]:
prompt = """
You will be given responses generated by a role-playing agent acting the character {agent_name}.
Here is the character seed data:
{persona_seed}
Your task is to rate the quality of the character generated interview responses using the specific criterion by following the evaluation steps. (on a scale of 1-7, with 1 being the worst). Give one score per interview question.
Interview Question:
Generated response:
Criterion:
Values: Does the response reflect the values and convictions of the character?
Evaluation Steps
1. Read through the profile and the values and convictions of the real character.
2. Read through the responses and identify the values and convictions of the agent.
3. Compare the responses to the profile. Looking for any consistencies or inconsistencies.
4. Rate the performance of how well response reflects the values and convictions of the character.
Evaluation Form (scores ONLY):
"""

In [18]:
import json

with open("./interview_transcripts/alberti_interview_transcript.json", "r") as f:
    interview_res = json.load(f)


print(interview_res[0:2])

[{'speaker': 'Interviewer', 'text': 'Some people save for big things, like a home, while others save for a rainy day. How about for you (in the last year)?'}, {'speaker': 'Interviewee', 'text': 'Mostly rainy day and future plans, if I’m being honest. I’m not anywhere near buying a home yet, so over the last year I’ve been trying to build up an emergency cushion and also set aside a little for bigger goals down the line. I’ve been pretty focused on saving for a car repair fund, unexpected bills, and eventually classes for hospitality or a manager certification.\n\nI also keep a small separate bucket for things I actually enjoy, like a weekend trip up to Door County or tickets for a Brewers game, but I’m pretty careful about that. I like knowing I’ve got a buffer more than spending just to spend.'}]


In [19]:
def transcript_to_qa(transcript):
    pairs = []

    for i in range(0, len(transcript) - 1, 2):
        q = transcript[i]
        a = transcript[i + 1]

        if q["speaker"] == "Interviewer" and a["speaker"] == "Interviewee":
            pairs.append({
                "Interview Question": q["text"],
                "Generated response": a["text"]
            })

    return pairs

interview_qa = transcript_to_qa(interview_res)

print(interview_qa[0])

{'Interview Question': 'Some people save for big things, like a home, while others save for a rainy day. How about for you (in the last year)?', 'Generated response': 'Mostly rainy day and future plans, if I’m being honest. I’m not anywhere near buying a home yet, so over the last year I’ve been trying to build up an emergency cushion and also set aside a little for bigger goals down the line. I’ve been pretty focused on saving for a car repair fund, unexpected bills, and eventually classes for hospitality or a manager certification.\n\nI also keep a small separate bucket for things I actually enjoy, like a weekend trip up to Door County or tickets for a Brewers game, but I’m pretty careful about that. I like knowing I’ve got a buffer more than spending just to spend.'}


In [20]:
from langchain.messages import HumanMessage, SystemMessage

pf = prompt.format(agent_name="Mary Alberti", persona_seed=json.dumps(persona_seed, indent=2))

res = llm.invoke([
    SystemMessage(content=pf),
    HumanMessage(content="Interview Transcript: " + json.dumps(interview_qa, indent=2))
])

print(res)

content='6\n7\n7\n6\n6\n7\n7\n6\n7\n6\n6\n6\n7\n7\n7\n7\n7\n6\n7\n6\n6' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 6869, 'total_tokens': 6913, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DOQzByJTydkryVsz0cXBh1hpSnKra', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d3550-bb4f-70f0-932a-0edacfd0bda0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6869, 'output_tokens': 44, 'total_tokens': 6913, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [21]:
with open("./interview_transcripts/validation_score.json", "w") as f:
    vals = res.content.split("\n")
    nums = [int(x) for x in vals if x.strip()]
    avg = sum(nums) / len(nums)

    
    data = {"interview_qa_score": avg, "interview_qa_all_scores": res.content} 
    json.dump(data, f, indent=2)